In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.feature_extraction.text import TfidfVectorizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)


In [ ]:
df_x = pd.read_csv('metadata.csv')
df_x.info()
display(df_x)

In [ ]:
# Ograniczamy metadata do wybranych miejsc
choosen_sites = ['Panther', 'Robin', 'Fox', 'Moose', 'Crow', 'Shrew', 'Mouse']

df_x = df_x[df_x['site_id'].isin(choosen_sites)]
display(df_x)

In [ ]:
# koordynaty wybranych miejsc

sites = df_x['site_id'].unique()
coords = []
for i in sites:
    temp = df_x.loc[df_x['site_id']==i]
    coords.append(i)
    coords.append(temp.iloc[0]['lat'])
    coords.append(temp.iloc[0]['lng'])
coords
#Fox - Phoenix
#Panther - Orlando
#Moose, Crow - Ottawa
#Robin, Shrew, Mouse - London
#dataframe dla każdego miasta (Ottawa - Moose and Crow, London - Robin, Shrew, Mouse)
#główny dataframe
#electricity, gas, steam, solar, weather

In [ ]:
# wybieramy które kolumny chcemy zostawić
df_y = df_x[['building_id', 'site_id', 'sqm', 'yearbuilt']].copy().reset_index(drop=True)
display(df_y)
df_y.info()

In [ ]:
# dataframe z pogodą, wybieramy miejsca, które nas interesują
df_w = pd.read_csv('weather.csv')
df_w = df_w[df_w['site_id'].isin(choosen_sites)]
df_w['timestamp'] = pd.to_datetime(df_w['timestamp'])
display(df_w)

In [ ]:
# zmieniamy podział czasowy na dni, liczę średnią z całego dnia dla wszystkich wartości (wydaję mi się, że to powinno być ok)
df_w_day = df_w.groupby([df_w['timestamp'].dt.date, 'site_id']).mean(numeric_only=True).reset_index()
df_w_day.set_index('timestamp', inplace=True)
df_w_day.info()
display(df_w_day)

In [ ]:
# wyciągam tylko jedno miejsce i przygotowuję dataframe'a tak, abym mógł to później połączyć z innymi dataframe'ami
# generalnie powinienem zrobić to bardziej elegancko, za pomocą pętli, ale już mi się dzisiaj nie chce kombinować :D 
df_w_crow = df_w_day.loc[df_w_day['site_id']=='Crow']
df_w_crow = df_w_crow.add_suffix("_crow")
df_w_crow = df_w_crow.drop(['site_id_crow'], axis=1)
display(df_w_crow)
df_w_crow.info() #721 rows

# no i tutaj jest też jeden problem, dla Crow i Moose (oba to Ottawa) brakuje 10 wierszy w porównaniu do reszty, później i tak to połączę po indexie (timestamp)
# jakaś dziura jest, trzeba uważać

In [ ]:
df_w_robin = df_w_day.loc[df_w_day['site_id']=='Robin']
df_w_robin = df_w_robin.add_suffix("_robin")
df_w_robin = df_w_robin.drop(['site_id_robin'], axis=1)
display(df_w_robin) #731 rows

In [ ]:
df_w_panther = df_w_day.loc[df_w_day['site_id']=='Panther']
df_w_panther = df_w_panther.add_suffix("_panther")
df_w_panther = df_w_panther.drop(['site_id_panther'], axis=1)
display(df_w_panther) #731 rows

In [ ]:
df_w_fox = df_w_day.loc[df_w_day['site_id']=='Fox']
df_w_fox = df_w_fox.add_suffix("_fox")
df_w_fox = df_w_fox.drop(['site_id_fox'], axis=1)
display(df_w_fox) #731 rows

In [ ]:
df_w_moose = df_w_day.loc[df_w_day['site_id']=='Moose']
df_w_moose = df_w_moose.add_suffix("_moose")
df_w_moose = df_w_moose.drop(['site_id_moose'], axis=1)
display(df_w_moose) #721 rows

In [ ]:
df_w_shrew = df_w_day.loc[df_w_day['site_id']=='Shrew']
df_w_shrew = df_w_shrew.add_suffix("_shrew")
df_w_shrew = df_w_shrew.drop(['site_id_shrew'], axis=1)
display(df_w_shrew) #731 rows

In [ ]:
df_w_mouse = df_w_day.loc[df_w_day['site_id']=='Mouse']
df_w_mouse = df_w_mouse.add_suffix("_mouse")
df_w_mouse = df_w_mouse.drop(['site_id_mouse'], axis=1)
display(df_w_mouse) #731 rows

In [ ]:
# Łączę dataframe'y za pomocą funkcji .concat, jest o tyle spoko, że dodaje po indexach, nic nie powinno się pomieszać
# jak będziecie chcieli połączyć dataframe'y w jakiejś innej konfiguracji, to polecam tym .concatem, tylko z tego co widzę to kolejność ma znaczenie
# gdy df_w_crow (gdzie jest mniej wierszy) dałem na pierwszym miejscu, to na koniec dodało mi te brakujące daty i kolejność lekko mi się pomieszała
df_weather_clean = pd.concat([df_w_panther, df_w_robin, df_w_crow, df_w_fox, df_w_moose, df_w_shrew, df_w_mouse], axis=1)
df_weather_clean.index.name = 'timestamp'
display(df_weather_clean)

In [ ]:
# robimy porządek z bazą, ustawiamy timestamp jako index i dodajemy _electricity do każdej kolumny
df_electricity = pd.read_csv('electricity.csv')
df_electricity = df_electricity.add_suffix("_electricity")
df_electricity['timestamp_electricity'] = pd.to_datetime(df_electricity['timestamp_electricity'])
df_electricity.set_index('timestamp_electricity', inplace=True)

df_electricity.info()
display(df_electricity)

In [ ]:
# z podziału godzinowego przechodzimy w dniowy, sumujemy wartości, wybieramy kolumny z miejscami, które nas interesują
df_electricity_day = df_electricity.resample('D').sum()
df_electricity_clean = df_electricity_day[df_electricity_day.columns[df_electricity_day.columns.str.contains('Panther|Robin|Fox|Moose|Crow|Shrew|Mouse', case=False)]].copy()
df_electricity_clean.info()
display(df_electricity_clean)

In [ ]:
# analogicznie jak z electricity
df_gas = pd.read_csv('gas.csv')
df_gas = df_gas.add_suffix("_gas")
df_gas['timestamp_gas'] = pd.to_datetime(df_gas['timestamp_gas'])
df_gas.set_index('timestamp_gas', inplace=True)
df_gas.info()
display(df_gas)

In [ ]:
df_gas_day = df_gas.resample('D').sum()
df_gas_clean = df_gas_day[df_gas_day.columns[df_gas_day.columns.str.contains('Panther|Robin|Fox|Moose|Crow|Shrew|Mouse', case=False)]].copy()
df_gas_clean.info()
display(df_gas_clean)

In [ ]:
df_steam = pd.read_csv('steam.csv')
df_steam = df_steam.add_suffix("_steam")
df_steam['timestamp_steam'] = pd.to_datetime(df_steam['timestamp_steam'])
df_steam.set_index('timestamp_steam', inplace=True)
df_steam.info()
display(df_steam)


In [ ]:
df_steam_day = df_steam.resample('D').sum()
df_steam_clean = df_steam_day[df_steam_day.columns[df_steam_day.columns.str.contains('Panther|Robin|Fox|Moose|Crow|Shrew|Mouse', case=False)]].copy()
df_steam_clean.info()
display(df_steam_clean)

In [ ]:
df_solar = pd.read_csv('solar.csv')
df_solar = df_solar.add_suffix("_solar")
df_solar['timestamp_solar'] = pd.to_datetime(df_solar['timestamp_solar'])
df_solar.set_index('timestamp_solar', inplace=True)
df_solar.info()
display(df_solar)

In [ ]:
# tu się okazuje, że jest pusto. no cóż
df_solar_day = df_solar.resample('D').sum()
df_solar_clean = df_solar_day[df_solar_day.columns[df_solar_day.columns.str.contains('Panther|Robin|Fox|Moose|Crow|Shrew|Mouse', case=False)]].copy()
df_solar_clean.info()
display(df_solar_clean)

In [ ]:
# połączone dataframe'y electricity, gas i steam
df_sources = pd.concat([df_electricity_clean, df_steam_clean, df_gas_clean], axis=1)
df_sources.index.name = 'timestamp'
display(df_sources)

In [ ]:
# z jakiegoś powodu łączenie .concatem mi tu nie działa (timestampy się dublują), więc użyję .join
df_sources_weather = df_sources.join(df_weather_clean)
display(df_sources_weather)

In [153]:
# Zapis do pliku csv
df_sources.to_csv('Sources.csv')
df_weather_clean.to_csv('Weather_clean.csv')
df_y.to_csv('meta.csv')
df_sources_weather.to_csv('Sources_and_weather.csv')

Nadal nie mam dobrego pomysłu jak połączyć ten ostatni dataframe z pierwszym, gdzie są m^2, rok budynku itd, żeby to jakoś wyglądało. Może jutro wpadnę na jakiś pomysł.
Ale chyba nadal będzie się dało jakoś wyliczyć eui bez łączenia w razie co.